In [ ]:
# | default_exp ui

In [ ]:
# | export
from fasthtml.common import *
from fasthtml.jupyter import JupyUvi, HTMX
from hyperrun.core import search_code, sp
from lisette import contents, AsyncChat
from dotenv import load_dotenv
from fhdaisy import *
from fastlucide import SvgSprites, SvgStyle
from litellm import completion_cost
from uuid import uuid4

load_dotenv()

icons = SvgSprites()
chats = {}
CHAT_CSS = (
    Style("""
.chat{min-width:0}
.chat-bubble{max-width:100%;min-width:0;overflow-wrap:anywhere;word-break:break-word}
.chat-bubble *{max-width:100%}
.chat-bubble pre{max-width:100%;overflow-x:auto;white-space:pre-wrap;overflow-wrap:anywhere;background:#1e293b;color:#e2e8f0;padding:.75em;border-radius:.5em;margin:.5em 0}
.chat-bubble code{font-size:.85em;overflow-wrap:anywhere;word-break:break-word}
.chat-bubble :not(pre)>code{background:#e2e8f0;padding:.1em .3em;border-radius:.25em}
.chat-bubble table{display:block;max-width:100%;overflow-x:auto}
.chat-bubble img{height:auto}
.chat-bubble ul{list-style:disc;padding-left:1.5em;margin:.5em 0}
.chat-bubble ol{list-style:decimal;padding-left:1.5em;margin:.5em 0}
.chat-bubble p{margin:.5em 0}
.chat-bubble h1,.chat-bubble h2,.chat-bubble h3{font-weight:700;margin:.75em 0 .25em}
.chat-bubble strong{font-weight:700}
.chat-bubble a{text-decoration:underline}
"""),
)
app, rt = fast_app(
    pico=False,
    hdrs=(
        *daisy_hdrs,
        CHAT_CSS,
        SvgStyle(),
        Script(src="https://unpkg.com/htmx-ext-sse@2.2.3/sse.js"),
        Script(src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"),
    ),
)


In [ ]:
# | export
def chat_content(standalone=False):
    box_cls = "w-full h-screen p-0 bg-base-100" if standalone else "modal-box w-11/12 max-w-4xl p-0 bg-base-100 border border-base-300 rounded-3xl shadow-[0_25px_80px_-20px_rgba(0,0,0,0.45)]"
    return Div(
        Div(cls="flex items-center justify-between px-6 py-5 border-b border-base-200")(
            Div(cls="flex items-center gap-3")(
                Div(
                    cls="w-10 h-10 rounded-2xl bg-primary/10 text-primary flex items-center justify-center"
                )(icons("sparkles", sz=20)),
                Div(
                    Div("Ask HyperRun", cls="font-semibold text-xl leading-tight"),
                    Div("Code-aware assistant", cls="text-sm text-base-content/60"),
                ),
            ),
            Div(cls="flex items-center gap-2")(
                Button(
                    icons("settings", sz=16),
                    cls="btn btn-ghost btn-sm btn-circle",
                    hx_get="/settings",
                    hx_target="#messages",
                    hx_swap="innerHTML",
                ),
                Button(
                    icons("trash-2", sz=16),
                    cls="btn btn-ghost btn-sm btn-circle",
                    hx_post="/clear-messages",
                    hx_target="#messages",
                    hx_swap="innerHTML",
                ),
                Label(
                    icons("x", sz=16),
                    fr="chat-modal",
                    cls="btn btn-ghost btn-sm btn-circle",
                ),
            ),
        ),
        Div(
            id="messages",
            cls="overflow-y-auto overflow-x-hidden h-[75vh] px-6 py-5 space-y-4 bg-base-100"
,
        ),
        Div(cls="px-6 py-5 border-t border-base-200 bg-base-100 rounded-b-3xl")(
            Form(
                cls="join w-full",
                hx_post=ask,
                hx_target="#messages",
                hx_swap="beforeend",
            )(
                Input(
                    id="query",
                    name="query",
                    placeholder="Ask a follow-up question...",
                    cls="input input-bordered join-item flex-1 h-14 text-base rounded-l-2xl",
                ),
                Button(
                    icons("send", sz=18),
                    cls="btn btn-primary join-item h-14 px-6 rounded-r-2xl",
                ),
            )
        ),
        cls=box_cls
    )


@rt("/chat", methods=["GET"])
def chat_panel():
    return chat_content()


@rt("/chat-standalone")
def chat_standalone():
    return Html(
        Head(
            *daisy_hdrs,
            SvgStyle(),
            CHAT_CSS,
            Script(src="https://unpkg.com/htmx.org"),
            Script(src="https://unpkg.com/htmx-ext-sse@2.2.3/sse.js"),
            Script(src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"),
        ),
        Body(icons, chat_content(standalone=True), cls="bg-transparent"),
    )


In [ ]:
# | export
from litellm import models_by_provider


@rt("/settings", methods=["GET"])
def settings_panel():
    providers = sorted(models_by_provider.keys())
    return Div(cls="p-4 space-y-4")(
        H3("Settings", cls="font-semibold text-lg"),
        Select(
            *[Option(p, value=p) for p in providers],
            id="provider",
            name="provider",
            cls="select select-bordered w-full",
            hx_get=model_options,
            hx_target="#model-select",
            hx_include="#provider",
        ),
        Div(id="model-select"),
        Input(
            id="api_key",
            name="api_key",
            type="password",
            placeholder="Enter API key...",
            cls="input input-bordered w-full",
        ),
        Button(
            "Save",
            cls="btn btn-primary w-full",
            hx_post=save_settings,
            hx_include="#provider,#model,#api_key",
            hx_target="#messages",
            hx_swap="innerHTML",
        ),
    )


@rt("/save-settings", methods=["POST"])
def save_settings(provider: str, model: str, api_key: str, sess):
    sess["provider"] = provider
    sess["model"] = model
    sess["api_key"] = api_key
    return Div(cls="p-4 text-center")(
        Span("✓ Settings saved!", cls="text-success font-semibold")
    )


@rt("/models", methods=["GET"])
def model_options(provider: str):
    models = models_by_provider.get(provider, [])
    return Select(
        *[Option(m, value=m) for m in models],
        id="model",
        name="model",
        cls="select select-bordered w-full",
    )


In [ ]:
# | export
@rt("/close-chat", methods=["GET"])
def close_chat():
    return Div()


@rt("/clear-messages", methods=["POST"])
def clear_messages(sess):
    sid = sess.get("session_id", None)
    if sid and sid in chats:
        del chats[sid]
    sess.pop("total_cost", None)
    return Div()


In [ ]:
# | export


def user_bubble(msg):
    return Div(cls="chat chat-end")(
        Div("You", cls="chat-header text-xs text-base-content/60 mb-1"),
        Div(
            msg,
            cls="chat-bubble chat-bubble-primary text-sm md:text-base max-w-[82%] shadow-sm",
        ),
    )


def ai_bubble_stream(query, msg_id: str, model_name: str, total_cost: float):
    return Div(cls="chat chat-start")(
        Div("HyperRun", cls="chat-header text-xs text-base-content/60 mb-1"),
        Div(
            Div(
                Span(cls="loading loading-dots loading-sm"),
                id=f"resp-{msg_id}",
                hx_ext="sse",
                sse_connect=stream_response.to(query=query, msg_id=msg_id),
                sse_swap="message",
                sse_close="close",
                hx_swap="beforeend",
                hx_on__sse_close="this.innerHTML=marked.parse(this.textContent)",
            ),
            Div(
                Span(model_name, cls="font-medium"),
                Span(" | Total ", cls="mx-1"),
                Span(f"${total_cost:.6f}", id=f"cost-{msg_id}"),
                cls="mt-2 text-[11px] text-base-content/60",
            ),
            cls="chat-bubble bg-base-200 text-base-content text-sm md:text-base max-w-[86%] shadow-sm break-words",
        ),
    )


@rt("/ask", methods=["POST"])
def ask(query: str, sess):
    model_name = sess.get("model", "claude-sonnet-4-20250514")
    total_cost = float(sess.get("total_cost", 0.0) or 0.0)
    msg_id = uuid4().hex[:10]
    new_inp = Input(
        id="query",
        name="query",
        placeholder="Ask a follow-up question...",
        cls="input input-bordered join-item flex-1 h-14 text-base rounded-l-2xl",
        hx_swap_oob="true",
    )
    return Div(
        user_bubble(query), ai_bubble_stream(query, msg_id, model_name, total_cost)
    ), new_inp


In [ ]:
# | export
@rt("/stream", methods=["GET"])
async def stream_response(query: str, msg_id: str, sess):
    sid = sess.get("session_id", None)
    if not sid:
        sid = str(uuid4())
        sess["session_id"] = sid

    if sid not in chats:
        model = sess.get("model", "claude-sonnet-4-20250514")
        api_key = sess.get("api_key", None)
        chats[sid] = AsyncChat(model, sp=sp, tools=[search_code], api_key=api_key)

    chat = chats[sid]

    async def generate():
        final_response = None
        async for chunk in await chat(query, stream=True):
            if hasattr(chunk, "choices") and hasattr(chunk.choices[0], "delta"):
                text = chunk.choices[0].delta.content
                if text:
                    yield sse_message(Span(text))
            elif hasattr(chunk, "usage"):
                final_response = chunk

        resp_cost = 0.0
        if final_response is not None:
            try:
                resp_cost = float(
                    completion_cost(completion_response=final_response) or 0.0
                )
            except Exception:
                resp_cost = float(getattr(final_response, "response_cost", 0.0) or 0.0)

        total_cost = float(sess.get("total_cost", 0.0) or 0.0) + resp_cost
        sess["total_cost"] = total_cost
        yield sse_message(
            Span(f"${total_cost:.6f}", id=f"cost-{msg_id}", hx_swap_oob="true")
        )
        yield sse_message(Span(""), event="close")

    return EventStream(generate())


In [ ]:
# | export
def chat_widget():
    return Div(
        icons,
        Label(
            Span(cls="inline-flex items-center gap-2")(
                icons("message-circle", sz=20),
                Span("Ask AI", cls="font-semibold tracking-tight"),
            ),
            fr="chat-modal",
            cls="btn btn-primary h-14 min-h-14 px-7 rounded-full fixed bottom-6 right-6 z-50 shadow-2xl text-base",
        ),
        Input(type="checkbox", id="chat-modal", cls="modal-toggle"),
        Div(cls="modal")(
            chat_content(),
            Label(cls="modal-backdrop", fr="chat-modal"),
        ),
    )

@rt("/")
def home():
    return Titled("HyperRun", chat_widget())


In [ ]:
#| export
from hyperrun.core import ensure_index
@app.on_event("startup")
async def startup(): await ensure_index()


In [ ]:
# | hide
server = JupyUvi(app)

In [ ]:
#| export 
serve()

In [ ]:
# | export
HTMX()